**Introductory and intermediate computing for Data Science [Barcelona School of Economics]**

`Instructor:` Maxim Fedotov  
`Program:` M.Sc. in Data Science Methodology

# Class 4

You will be very often working with several parameter values that you want to try. In machine learning it is a regular practice to do a "hyperparameter tuning", i.e. testing quality of your fit (based on some quantitative metric like MSE, MAE, ROC, AUC and so on) with respect to different values of your algorithm. 

Let's consider a bit simplified example: we want to compute a log-likelihood for a random sample that we have. We will be using `numpy` package here to get a random sample (see the class_5 notebook). In fact, it would be better to use `numpy.array` instead of the one-dimensional `x` to vectorize computations (and get computation time gains), but for educational purposes we avoid that. Just keep it in mind.

In addition to that, you sometimes need to *map* functions on collections of objects, and be able to write some small technical functions in a more concise way than a regular definition. We will also consider it in this notebook.

## Handling functions: map, lambda, functools.partial
First, define a function that computes the logarithm of the normal pdf with parameters `mu` and `sigma` at point `x`.

In [ ]:
from math import pi, log
import numpy

numpy.random.seed(1337)  # for reproducibility

def log_normal_pdf(x: float, mu: float = 0, sigma: float = 1) -> float:
    """
        Computes the natural logarithm of the pdf of a normal distribution
        with parameters mu and sigma at point x.
        
        args:
            x:         concrete point
            mu:        mean parameter of the distribution
            sigma:     st.dev. parameter of the distribution
        
        returns:
            The log-density at x.
    """
    if sigma <= 0:
        raise ValueError("The parameter sigma has to be positive.")
    return -(x - mu)**2 / (2 * sigma**2) - 0.5 * log(2 * pi * sigma**2)

log_normal_pdf(0)

### Map
Suppose now that we have a sample from an actual normal distribution. I convert it to a list for educational purposes of the current class since we have not seen `numpy` yet. In reality, you would try to use `numpy` objects and operations for such tasks (although difference is just in vectorizing operations, not in a conceptual approach).

In [ ]:
x_sample = list(numpy.random.normal(loc=1, scale=2, size=100))
x_sample[:10]

In [ ]:
log_pdf_map = map(log_normal_pdf, x_sample)
log_pdf_map

Notice that the result has a specific type. However, it is still iterable (but not subscriptable), so you can transform it to some more tractable sequence type, or you can just iterate through it using a `for` loop.

In [ ]:
log_pdf_sample = list(log_pdf_map)  # after this, log_pdf_map is exhausted, see the comment below
log_pdf_sample[:10]

Note that map returns a *lazy iterator*: it does not store the results, it produces them one at a time as you ask for them. Once you have iterated over it, it is exhausted, and a second pass gives you an empty result. You can check this by running the Python cell below. It will print nothing because the `list(...)` funciton has already passed over the values in `log_pdf_map` to create a list, and thus `log_pdf_map` cannot be iterated over again. 

In [ ]:
for result in log_pdf_map:
    print(result)

### Lambda

Sometimes you need a tool to make up a function on the go to use it for some technical purposes rather than a "base" function for your computations. In this case, you can use a `lambda` function definition. The syntax is the following:

```{Python}
lambda arg1, arg2 ...: code line
```

It is used to create one-line functions. You can also assign it to a variable, so you will be able to call it later.

Suppose you want a small utility to transform a feature in your data. You can write it concisely like this:

In [ ]:
treatment_indicator = lambda x: 1 if x == 'treatment' else 0

list(
    map(treatment_indicator, ["treatment", "control", "treatment", "control"])
)

A note on style: assigning a `lambda` to a variable, as above, works, but it is the one use of `lambda` that the official style guide advises against. If the function is going to have a name, a regular `def` is clearer and gives you better error messages:

```{Python}
def treatment_indicator(x):
    return 1 if x == 'treatment' else 0
```

Where `lambda` genuinely earns its place is the anonymous, inline case: a small function that you pass directly to something else and never need to refer to again. That is exactly the situation we are in now.

In our case, we want to be able to *map* our function `log_normal_pdf` to a list, and be able to set up the parameters `mu` and `sigma`. The built-in `map(...)` does not give us such functionality. So, we can try using `lambda` in combination with `map(...)` to achieve that.

In [ ]:
log_pdf_map_customized = list(
    map(lambda x: log_normal_pdf(x, mu=1, sigma=2), x_sample)
)
log_pdf_map_customized[:10]

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('ggplot')

indices = numpy.argsort(x_sample)
plt.figure(figsize=(5, 3.5))
plt.scatter(numpy.array(x_sample)[indices], numpy.array(log_pdf_map_customized)[indices], s=6)
plt.title(r"Normal log-density with $\mu=1$ and $\sigma=2$ at each point in the sample.", size=10)
plt.xlabel("x", size=9)
plt.ylabel(r"$\log \phi (x\ |\ \mu, \sigma)$", size=9)
plt.show()

Another way of mapping a function with multiple arguments onto a collection (or several collections) while holding all other parameters fixed is to use a function named `partial` in the `functools` package. The main difference is that `functools.partial` yields a specific `partial` object, and `lambda` is treated like a usual Python function.

In [ ]:
import functools

list(
    map(functools.partial(log_normal_pdf, mu=1, sigma=2), x_sample)
)[:10]

Now we have the pieces we need. Let's define a function that computes the log-likelihood, which in this case is:
$$l(\mathbf{x}\ |\ \mu, \sigma^2) = \sum_{i = 1}^{n} \log\phi(x_i\ |\ \mu, \sigma^2)$$
where $\mathbf{x} = (x_1, \ldots, x_n)^T$ and $\phi(\cdot\ |\ \mu, \sigma^2)$ is the p.d.f. of a normal random variable with mean $\mu$ and variance $\sigma^2$.

In practice you would do this with `numpy`, and I recommend using `map` only when vectorization is not possible at all. Here we map `log_normal_pdf` over the list and sum the results.

In [ ]:
def log_likelihood_normal(x: list, mu: float = 0, sigma: float = 1) -> float:
    log_pdf_individual = list(
        map(lambda xi: log_normal_pdf(xi, mu=mu, sigma=sigma), x)
    )
    return sum(log_pdf_individual)

In [ ]:
log_likelihood_normal(x_sample, 1, 2)

## Handling parameters: zip and itertools (and not only)

Now, we are moving towards exploring target values with respect to multiple values of parameters. That is often the case in ML applications. To specify a range of possible parameters you can use the built-in function `zip(iterable1, iterable2...)`. The iterables passed should be of the same size. It produces a `zip` object which consists of tuples. The size of each tuple is the number of the passed iterables.

In [ ]:
list(
    zip([1, 2, 3], [4, 5, 6])
)

In our case, we could explore different parameter combinations with `zip(...)` like that:

In [ ]:
mu_vals = [2] * 3 + [-2] * 3
sigma_vals = [1, 2, 3] * 2

for mu, sigma in zip(mu_vals, sigma_vals):
    log_pdf = log_likelihood_normal(x_sample, mu, sigma)
    print(f"The normal log-likelihood of the sample with mu = {mu}, sigma = {sigma} is {log_pdf}")

However, we do not explore a lot of parameters like that. So, it is often the case that you wish to use a *parameters grid*. To do that, you can use `product` function from the `itertools` package. It will give us a whole set of possible combinations (of a size of the number of passed iterables). You can think of it as a set product to some extent. Now, the iterables that we want to use to produce a grid do not have to be of the same size.

In [ ]:
from itertools import product

mu_vals = range(-2, 5)
sigma_vals = range(1, 5)

param_grid = list(product(mu_vals, sigma_vals))

# let's store all the results
log_likelihood_vals = list(
    map(lambda par_tuple: log_likelihood_normal(x_sample, *par_tuple), param_grid)
)

index_max_loglik = numpy.argmax(log_likelihood_vals)
print("Parameters that give the best (in the specified range) log-likelihood:", param_grid[index_max_loglik])

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(
    [par_tuple[0] for par_tuple in param_grid], 
    [par_tuple[1] for par_tuple in param_grid], 
    c=log_likelihood_vals, cmap='Greens'
)
plt.title(r"The log-likelihood magnitude w.r.t. $\mu$ and $\sigma$", size=10)
plt.xlabel(r"$\mu$", size=9)
plt.ylabel(r"$\sigma$", size=9)
plt.show()

# as we can see, log-likelihood values in this region are pretty close to each other.